##### Imports and constants

In [41]:
import azure.functions as func
import logging
import json
import urllib.request
import pandas as pd
import numpy as np
import openpyxl

import re

In [117]:
#change mappings according to the final form (change left side)
mapping = {"Completion time":"created_at",
           "What is your Telegram Handle? (eg. @ilovecefest)\n": "tele_handle",
           "What is your full name?": "full_name",
           "What house are you from ? ": "house",
           "I am currently... (in AY26/27)" : "year",
            "Email": "student_email",
            "First Choice ": "trail_first_choice",
            "Second Choice ": "trail_second_choice",
            "Third Choice ": "trail_third_choice",
            'Why did you select your chosen trails ? (EG: T1 because .....) ': "writeup"}
path = "C:\\Users\\xuan2\\Downloads\\_CE Fest 2026 Participant Sign-up Form!(1-1).xlsx"

##### Function for sending https req to firebase

In [43]:
ENDPOINT = "https://inpro-participants-submit-3avzbu5vaa-as.a.run.app"

def upload_data(payload : dict): #function to upload data onto firebase (DONT NEED TOUCH)
    data = json.dumps(payload).encode()
    req = urllib.request.Request(
        ENDPOINT,
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req) as resp:
            body = resp.read().decode()
            print(f"Status: {resp.status}")
            print(f"Response: {body}")
    except urllib.error.HTTPError as e:
        body = e.read().decode()
        print(f"Status: {e.code}")
        print(f"Error: {body}")

##### Trial Data

In [118]:
def normalize(s):
    s = '' if s is None else str(s)
    # remove BOM, zero-width, replace newlines & NBSP, collapse spaces
    s = s.replace('\ufeff', '').replace('\u200b','').replace('\xa0',' ')
    s = s.replace('\r',' ').replace('\n',' ')
    s = re.sub(r'\s+', ' ', s).strip()
    return s

# read the sheet (specify header row if needed, e.g. header=1)
raw = pd.read_excel(path)

def rename(mapping : dict, raw: pd.DataFrame):
    cols_norm = [normalize(c) for c in raw.columns] #normaliising the cols of df
    norm_map = {normalize(k): v for k, v in mapping.items()} #create updated mappings
    raw.columns = cols_norm
    raw = raw.rename(columns=norm_map)
    return raw

renamed = rename(mapping, raw)
print(renamed.tail())

   ID          Start time          created_at       student_email        Name  \
0   1 2026-05-22 13:27:07 2026-05-22 13:29:45  e1525929@u.nus.edu  Brian Pang   

   Last modified time        2   full_name tele_handle  \
0                 NaN  I agree  Brian Pang    @vruaaan   

  What is your NUS E......@u.edu.nus email ?     year    house  \
0                         e1525929@u.nus.edu  Year 2   Phoenix   

  What is your gender ?          What is your faculty and major?  \
0                  Male  School of Computing, Business Analytics   

       trail_first_choice          trail_second_choice  \
0  T6 - No Fixed Address   T8 - Our Home, Our Stories    

        trail_third_choice                   writeup  
0  T7 - Bridges of belief   testtesttesttesttesttest  


##### Function to read 1 row and process accordingly to make it same format as payload


In [121]:
def parse_numeric_from_string(v):
    # return None for NaN-like values
    if pd.isna(v):
        return None
    if isinstance(v, (int, float)):
        try:
            return int(v)
        except Exception:
            pass
    s = str(v)
    m = re.search(r'(\d+)', s)
    return int(m.group(1)) if m else s


def format_vals(init: dict) -> dict:
    if 'year' in init:
        init['year'] = parse_numeric_from_string(init['year'])
    for k in ('trail_first_choice', 'trail_second_choice', 'trail_third_choice'):
        if k in init:
            init[k] = parse_numeric_from_string(init[k])
    return init


def normalize_value(v):
    if pd.isna(v):
        return None
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, pd.Timestamp):
        return v.to_pydatetime()
    if isinstance(v, (np.floating,)):
        return float(v)
    if isinstance(v, (np.bool_,)):
        return bool(v)
    return v


def process_row(srs: pd.Series) -> dict:
    """Accepts a pandas Series (one row) and returns a JSON-serializable dict."""
    cols = ['created_at', 'tele_handle', 'full_name', 'house', 'student_email', 'year', 'status', 'role', 'trail_first_choice', 'trail_second_choice', 'trail_third_choice'] #edit col names according to the final one
    d = {}
    for col_name in cols:
        if col_name == 'role':
            val = srs.get('role')
            d['role'] = val if not pd.isna(val) else "Participant"
        elif col_name == 'status':
            val = srs.get('status')
            d['status'] = val if not pd.isna(val) else 'pending'
        else:
            d[col_name] = srs.get(col_name)
    d = format_vals(d)
    # normalize numpy / pandas types to Python builtins for JSON
    for k in list(d.keys()):
        d[k] = normalize_value(d[k])
    return d

In [122]:
read = df.tail(1)
print(read)
print(process_row(read.iloc[0]))

   ID          Start time     Completion time       student_email        Name  \
0   1 2026-05-22 13:27:07 2026-05-22 13:29:45  e1525929@u.nus.edu  Brian Pang   

   Last modified time        2   full_name tele_handle  \
0                 NaN  I agree  Brian Pang    @vruaaan   

  What is your NUS E......@u.edu.nus email ?      year  \
0                          e1525929@u.nus.edu  Year 2    

  What house are you from ?  What is your gender ?   \
0                    Phoenix                   Male   

           What is your faculty and major?           First Choice   \
0  School of Computing, Business Analytics  T6 - No Fixed Address    

                Second Choice             Third Choice   \
0  T8 - Our Home, Our Stories   T7 - Bridges of belief    

  Why did you select your chosen trails ? (EG: T1 because .....)   
0                           testtesttesttesttesttest               
{'created_at': None, 'tele_handle': '@vruaaan', 'full_name': 'Brian Pang', 'house': None, 'stude

##### Function to check form, hooked up to azure timer

In [ ]:
def load_counter(): #counter needs to be saved to cell "A1" of the second sheet labelled "counter"
    wb = openpyxl.load_workbook(path)
    if "counter" not in wb.sheetnames:
        ws = wb.create_sheet("counter")
        ws["A1"] = 0
        wb.save(path)
        return 0
    val = wb["counter"]["A1"].value
    return int(val) if val is not None else 0

def save_counter(value: int):
    wb = openpyxl.load_workbook(path)
    if "counter" not in wb.sheetnames:
        wb.create_sheet("counter")
    wb["counter"]["A1"] = value
    wb.save(path)

counter = load_counter()
#@app.timer_trigger(schedule="0 * * * * *", arg_name="myTimer", run_on_startup=False) #UNCOMMENT LATER
def check_forms(myTimer: func.TimerRequest) -> None:
    global counter
    # logging.info("Python timer trigger function executed.")
    if myTimer.past_due: # for tracking if the timer is due on time  
        logging.info("The timer is past due.")
    df = pd.read_excel(path, sheet_name="response").rename(columns=mapping)
    curr_num = len(df)
    if counter >= curr_num:
        return
    backlog = df.iloc[counter:curr_num]
    for _, row in backlog.iterrows():
        payload = process_row(row)
        upload_data(payload)
        counter += 1
        save_counter(counter)